# MSE 232 Project - Group 13
## Optimizing the 2026 FIFA World Cup Group-Stage Schedule

# 1. Data Import and Preparation

## This section imports the datasets and creates the sets and parameters used in the optimization models. The project data is stored in separate CSV files so that the optimization model can be updated without changing the model formulation.

In [ ]:
# libraries
import pandas as pd
import numpy as np
import gurobipy as gp
from gurobipy import GRB
import matplotlib.pyplot as plt

# csv files
teams = pd.read_csv("teams.csv")
venues = pd.read_csv("venues.csv")
matches = pd.read_csv("matches.csv")
travel = pd.read_csv("travel.csv")
weather = pd.read_csv("weather.csv")
audience_regions = pd.read_csv("audience_regions.csv")

In [ ]:
# data inspection

print("Teams:", teams.shape)
print("Venues:", venues.shape)
print("Matches:", matches.shape)
print("Travel:", travel.shape)
print("Weather:", weather.shape)
print("Audience Regions:", audience_regions.shape)

display(teams.head())
display(venues.head())
display(matches.head())
display(travel.head())

print("Number of teams:", teams["team_id"].nunique())
print("Number of venues:", venues["venue_id"].nunique())
print("Number of matches:", matches["match_id"].nunique())
print("Number of travel combinations:", len(travel))

In [ ]:
# data formatting

matches["match_date"] = pd.to_datetime(matches["match_date"])
weather["date"] = pd.to_datetime(weather["date"])

teams["team_id"] = teams["team_id"].astype(int)
venues["venue_id"] = venues["venue_id"].astype(int)

matches["match_id"] = matches["match_id"].astype(int)
matches["team_1_id"] = matches["team_1_id"].astype(int)
matches["team_2_id"] = matches["team_2_id"].astype(int)
matches["actual_venue_id"] = matches["actual_venue_id"].astype(int)

travel["team_id"] = travel["team_id"].astype(int)
travel["venue_id"] = travel["venue_id"].astype(int)

In [ ]:
# defining sets

Teams = teams["team_id"].tolist()
Venues = venues["venue_id"].tolist()
Matches = matches["match_id"].tolist()

Groups = sorted(matches["group"].unique().tolist())
Dates = sorted(matches["match_date"].unique().tolist())

Times = list(range(10, 23))

In [ ]:
# derive actual group-stage round from each team's match sequence

round_assignment = {}

for i in Teams:

    matches_i = matches[
        (matches["team_1_id"] == i)
        | (matches["team_2_id"] == i)
    ].copy()

    matches_i = matches_i.sort_values(
        ["match_date", "match_id"]
    )

    for r, m in enumerate(
        matches_i["match_id"].tolist(),
        start=1
    ):

        if m in round_assignment:

            if round_assignment[m] != r:
                raise ValueError(
                    "Inconsistent round assignment for match "
                    + str(m)
                )

        else:
            round_assignment[m] = r


matches["group_round"] = (
    matches["match_id"]
    .map(round_assignment)
    .astype(int)
)

In [ ]:
# match parameters

match_group = dict(zip(
    matches["match_id"],
    matches["group"]
))

group_round = dict(zip(
    matches["match_id"],
    matches["group_round"]
))

team_1 = dict(zip(
    matches["match_id"],
    matches["team_1_id"]
))

team_2 = dict(zip(
    matches["match_id"],
    matches["team_2_id"]
))

match_date = dict(zip(
    matches["match_id"],
    matches["match_date"]
))

actual_venue = dict(zip(
    matches["match_id"],
    matches["actual_venue_id"]
))

actual_kickoff_utc = dict(zip(
    matches["match_id"],
    matches["actual_kickoff_utc"]
))

In [ ]:
# team parameters

team_name = dict(zip(
    teams["team_id"],
    teams["team_name"]
))

base_camp_utc_offset = dict(zip(
    teams["team_id"],
    teams["base_camp_utc_offset"]
))

In [ ]:
# venue parameters

venue_name = dict(zip(
    venues["venue_id"],
    venues["venue_name"]
))

venue_city = dict(zip(
    venues["venue_id"],
    venues["city"]
))

venue_country = dict(zip(
    venues["venue_id"],
    venues["country"]
))

venue_utc_offset = dict(zip(
    venues["venue_id"],
    venues["utc_offset"]
))

venue_capacity = dict(zip(
    venues["venue_id"],
    venues["capacity"]
))

In [ ]:
# travel parameters

# duplicate travel check
duplicate_travel = travel.duplicated(
    subset=["team_id", "venue_id"]
).sum()

print("Duplicate travel combinations:", duplicate_travel)

travel_time = {}

for _, row in travel.iterrows():
    i = row["team_id"]
    v = row["venue_id"]

    travel_time[(i, v)] = row["total_time_travelled_hours"]

travel_distance = {}

for _, row in travel.iterrows():
    i = row["team_id"]
    v = row["venue_id"]

    travel_distance[(i, v)] = row["total_distance_travelled_kilometres"]

# missing travel check

missing_travel = []

for i in Teams:
    for v in Venues:
        if (i, v) not in travel_time:
            missing_travel.append((i, v))

print("Missing travel combinations:", len(missing_travel))

In [ ]:
# helper sets

# matches by date

matches_by_date = {}

for d in Dates:
    matches_by_date[d] = []

    for m in Matches:
        if match_date[m] == d:
            matches_by_date[d].append(m)

# matches for each team
team_matches = {}

for i in Teams:
    team_matches[i] = []

    for m in Matches:
        if team_1[m] == i or team_2[m] == i:
            team_matches[i].append(m)

for i in Teams:
    team_matches[i] = sorted(
        team_matches[i],
        key=lambda m: match_date[m]
    )

# matches by round
Round1Matches = []
Round2Matches = []
Round3Matches = []

for m in Matches:

    if group_round[m] == 1:
        Round1Matches.append(m)

    elif group_round[m] == 2:
        Round2Matches.append(m)

    elif group_round[m] == 3:
        Round3Matches.append(m)

# matches by group and round
matches_by_group_round = {}

for g in Groups:
    for r in [1, 2, 3]:

        matches_by_group_round[(g, r)] = []

        for m in Matches:
            if match_group[m] == g and group_round[m] == r:
                matches_by_group_round[(g, r)].append(m)

# 2. Metric Preparation and Evaluation Parameters

## This section defines the additional parameters, functions, and lookup tables used to evaluate both the FIFA schedule and the optimized schedules. Defining these measures before evaluating any schedule ensures that FIFA and all optimization models are assessed using the same methodology.

In [ ]:
# additional venue parameters

weather_protected = dict(zip(
    venues["venue_id"],
    venues["weather_protected"]
))

roof_type = dict(zip(
    venues["venue_id"],
    venues["roof_type"]
))

In [ ]:
# heat index function

def calculate_heat_index(temp_f, humidity):

    simple_hi = 0.5 * (
        temp_f
        + 61
        + 1.2 * (temp_f - 68)
        + 0.094 * humidity
    )

    simple_hi = (simple_hi + temp_f) / 2

    if simple_hi < 80:
        return simple_hi

    hi = (
        -42.379
        + 2.04901523 * temp_f
        + 10.14333127 * humidity
        - 0.22475541 * temp_f * humidity
        - 0.00683783 * temp_f**2
        - 0.05481717 * humidity**2
        + 0.00122874 * temp_f**2 * humidity
        + 0.00085282 * temp_f * humidity**2
        - 0.00000199 * temp_f**2 * humidity**2
    )

    return hi

In [ ]:
# heat index for weather data

weather["heat_index_f"] = weather.apply(
    lambda row: calculate_heat_index(
        row["temperature_f"],
        row["humidity_pct"]
    ),
    axis=1
)

In [ ]:
# weather lookup

weather_lookup = {}

for _, row in weather.iterrows():

    key = (
        row["venue_id"],
        row["date"],
        row["hour_local"]
    )

    weather_lookup[key] = row["heat_index_f"]

duplicate_weather = weather.duplicated(
    subset=["venue_id", "date", "hour_local"]
).sum()

print("Duplicate weather combinations:", duplicate_weather)

In [ ]:
# time zone difference lookup

timezone_difference = {}

for i in Teams:
    for v in Venues:

        timezone_difference[(i, v)] = abs(
            base_camp_utc_offset[i]
            - venue_utc_offset[v]
        )

# 3. FIFA Schedule Baseline Analysis
## This section evaluates FIFA's actual group-stage schedule using the common metrics defined above. Because the FIFA venue and kickoff assignments are fixed, no optimization is performed in this section. The existing FIFA schedule is evaluated directly and used as the benchmark for comparison with the optimized schedules.

In [ ]:
# constructing fifa schedule

fifa_schedule = matches[
    [
        "match_id",
        "group",
        "group_round",
        "team_1_id",
        "team_2_id",
        "match_date",
        "actual_venue_id",
        "actual_kickoff_utc"
    ]
].copy()

fifa_schedule["team_1_name"] = fifa_schedule["team_1_id"].map(team_name)
fifa_schedule["team_2_name"] = fifa_schedule["team_2_id"].map(team_name)

fifa_schedule["venue_name"] = fifa_schedule["actual_venue_id"].map(venue_name)
fifa_schedule["venue_city"] = fifa_schedule["actual_venue_id"].map(venue_city)
fifa_schedule["venue_country"] = fifa_schedule["actual_venue_id"].map(venue_country)

In [ ]:
# adding world cup 2026 actual kickoff times

def create_utc_kickoff_datetime(local_date, utc_time, utc_offset):

    # make sure kickoff time is a Python time object
    if isinstance(utc_time, str):
        utc_time = pd.to_datetime(
            utc_time,
            format="%H:%M"
        ).time()

    utc_minutes = utc_time.hour * 60 + utc_time.minute

    local_minutes = (
        utc_minutes + int(utc_offset * 60)
    ) % (24 * 60)

    local_hour = local_minutes // 60
    local_minute = local_minutes % 60

    local_datetime = (
        pd.Timestamp(local_date)
        + pd.Timedelta(hours=local_hour)
        + pd.Timedelta(minutes=local_minute)
    )

    utc_datetime = (
        local_datetime
        - pd.Timedelta(hours=utc_offset)
    )

    return utc_datetime


fifa_schedule["kickoff_datetime_utc"] = fifa_schedule.apply(
    lambda row: create_utc_kickoff_datetime(
        row["match_date"],
        row["actual_kickoff_utc"],
        venue_utc_offset[row["actual_venue_id"]]
    ),
    axis=1
)

In [ ]:
# fifa schedule travel analysis

fifa_total_travel_time = 0

for m in Matches:

    i = team_1[m]
    j = team_2[m]
    v = actual_venue[m]

    fifa_total_travel_time += 2 * travel_time[(i, v)]
    fifa_total_travel_time += 2 * travel_time[(j, v)]

print(
    "FIFA total team travel time:",
    round(fifa_total_travel_time, 2),
    "hours"
)

In [ ]:
# fifa travel by team
fifa_team_travel = {}

for i in Teams:

    fifa_team_travel[i] = 0

    for m in team_matches[i]:

        v = actual_venue[m]

        fifa_team_travel[i] += (
            2 * travel_time[(i, v)]
        )

fifa_team_travel_df = pd.DataFrame({
    "team_id": Teams,
    "team_name": [team_name[i] for i in Teams],
    "total_travel_time_hours": [
        fifa_team_travel[i]
        for i in Teams
    ]
})

In [ ]:
# summary metrics for fifa schedule

fifa_avg_team_travel = (
    fifa_team_travel_df["total_travel_time_hours"].mean()
)

fifa_max_team_travel = (
    fifa_team_travel_df["total_travel_time_hours"].max()
)

fifa_min_team_travel = (
    fifa_team_travel_df["total_travel_time_hours"].min()
)

fifa_std_team_travel = (
    fifa_team_travel_df["total_travel_time_hours"].std(ddof=0)
)

In [ ]:
# fifa rest and effective recovery

fifa_recovery_rows = []

for i in Teams:

    matches_i = team_matches[i]

    for k in range(len(matches_i) - 1):

        previous_match = matches_i[k]
        next_match = matches_i[k + 1]

        previous_row = fifa_schedule[
            fifa_schedule["match_id"] == previous_match
        ].iloc[0]

        next_row = fifa_schedule[
            fifa_schedule["match_id"] == next_match
        ].iloc[0]

        previous_kickoff = previous_row["kickoff_datetime_utc"]
        next_kickoff = next_row["kickoff_datetime_utc"]

        raw_rest_hours = (
            next_kickoff - previous_kickoff
        ).total_seconds() / 3600

        previous_venue = previous_row["actual_venue_id"]
        next_venue = next_row["actual_venue_id"]

        recovery_travel_time = (
            travel_time[(i, previous_venue)]
            + travel_time[(i, next_venue)]
        )

        effective_recovery_hours = (
            raw_rest_hours
            - recovery_travel_time
        )

        fifa_recovery_rows.append({
            "team_id": i,
            "team_name": team_name[i],
            "previous_match": previous_match,
            "next_match": next_match,
            "raw_rest_hours": raw_rest_hours,
            "recovery_travel_time_hours": recovery_travel_time,
            "effective_recovery_hours": effective_recovery_hours
        })

fifa_recovery = pd.DataFrame(fifa_recovery_rows)

In [ ]:
# summary metrics for fifa recovery

fifa_min_raw_rest = fifa_recovery["raw_rest_hours"].min()
fifa_avg_raw_rest = fifa_recovery["raw_rest_hours"].mean()

fifa_min_effective_recovery = (
    fifa_recovery["effective_recovery_hours"].min()
)

fifa_avg_effective_recovery = (
    fifa_recovery["effective_recovery_hours"].mean()
)

fifa_recovery["meets_72_hour_raw_rest"] = (
    fifa_recovery["raw_rest_hours"] >= 72
)

print("Minimum FIFA raw rest:", round(fifa_min_raw_rest, 2))
print("Average FIFA raw rest:", round(fifa_avg_raw_rest, 2))

print(
    "Minimum FIFA effective recovery:",
    round(fifa_min_effective_recovery, 2)
)

print(
    "Average FIFA effective recovery:",
    round(fifa_avg_effective_recovery, 2)
)

print(
    "Intervals below 72 hours raw rest:",
    (~fifa_recovery["meets_72_hour_raw_rest"]).sum()
)

print("Recovery intervals evaluated:", len(fifa_recovery))

if len(fifa_recovery) != 96:
    raise ValueError("Expected 96 FIFA recovery intervals.")

In [ ]:
# fifa schedule time zone analysis

fifa_timezone_rows = []

for m in Matches:

    v = actual_venue[m]

    for i in [team_1[m], team_2[m]]:

        fifa_timezone_rows.append({
            "match_id": m,
            "team_id": i,
            "team_name": team_name[i],
            "venue_id": v,
            "timezone_difference_hours":
                timezone_difference[(i, v)]
        })

fifa_timezone = pd.DataFrame(fifa_timezone_rows)

fifa_avg_timezone_difference = (
    fifa_timezone["timezone_difference_hours"].mean()
)

fifa_max_timezone_difference = (
    fifa_timezone["timezone_difference_hours"].max()
)

In [ ]:
# fifa schedule weather analysis

def utc_time_to_local_hour(utc_time, utc_offset):

    # convert string to Python time object if needed
    if isinstance(utc_time, str):
        utc_time = pd.to_datetime(
            utc_time,
            format="%H:%M"
        ).time()

    utc_minutes = utc_time.hour * 60 + utc_time.minute

    local_minutes = (
        utc_minutes + int(utc_offset * 60)
    ) % (24 * 60)

    local_hour = local_minutes // 60

    return int(local_hour)


# calculate local kickoff hour for each FIFA match
fifa_schedule["kickoff_local_hour"] = fifa_schedule.apply(
    lambda row: utc_time_to_local_hour(
        row["actual_kickoff_utc"],
        venue_utc_offset[row["actual_venue_id"]]
    ),
    axis=1
)


# check that all required weather observations exist
missing_fifa_weather = []

for _, row in fifa_schedule.iterrows():

    v = row["actual_venue_id"]
    d = row["match_date"]
    h = row["kickoff_local_hour"]

    if (v, d, h) not in weather_lookup:
        missing_fifa_weather.append((v, d, h))

print("Missing FIFA weather lookups:", len(missing_fifa_weather))

if len(missing_fifa_weather) > 0:
    raise ValueError("Missing weather data for FIFA schedule.")


# calculate FIFA weather exposure
fifa_weather_rows = []

for _, row in fifa_schedule.iterrows():

    m = row["match_id"]
    v = row["actual_venue_id"]
    d = row["match_date"]
    h = row["kickoff_local_hour"]

    heat_index = weather_lookup[(v, d, h)]

    if weather_protected[v] == 1:
        weather_exposure = 0
    else:
        weather_exposure = heat_index

    fifa_weather_rows.append({
        "match_id": m,
        "venue_id": v,
        "venue_name": venue_name[v],
        "date": d,
        "local_hour": h,
        "heat_index_f": heat_index,
        "weather_protected": weather_protected[v],
        "weather_exposure": weather_exposure
    })


fifa_weather = pd.DataFrame(fifa_weather_rows)

fifa_total_weather_exposure = (
    fifa_weather["weather_exposure"].sum()
)

fifa_avg_weather_exposure = (
    fifa_weather["weather_exposure"].mean()
)

fifa_max_weather_exposure = (
    fifa_weather["weather_exposure"].max()
)

In [ ]:
# fifa schedule seating opportunity

fifa_seating_opportunity = 0

for m in Matches:
    v = actual_venue[m]
    fifa_seating_opportunity += venue_capacity[v]

In [ ]:
# fifa schedule venue and country usage

fifa_venue_usage = {}

for v in Venues:
    fifa_venue_usage[v] = 0

for m in Matches:
    v = actual_venue[m]
    fifa_venue_usage[v] += 1

fifa_country_usage = {}

for m in Matches:

    v = actual_venue[m]
    country = venue_country[v]

    if country not in fifa_country_usage:
        fifa_country_usage[country] = 0

    fifa_country_usage[country] += 1

In [ ]:
# fifa schedule baseline summary

fifa_baseline = {
    "total_travel_time_hours": fifa_total_travel_time,
    "average_team_travel_time_hours": fifa_avg_team_travel,
    "maximum_team_travel_time_hours": fifa_max_team_travel,
    "minimum_team_travel_time_hours": fifa_min_team_travel,
    "travel_time_std_hours": fifa_std_team_travel,

    "minimum_raw_rest_hours": fifa_min_raw_rest,
    "average_raw_rest_hours": fifa_avg_raw_rest,

    "minimum_effective_recovery_hours":
        fifa_min_effective_recovery,

    "average_effective_recovery_hours":
        fifa_avg_effective_recovery,

    "average_timezone_difference_hours":
        fifa_avg_timezone_difference,

    "maximum_timezone_difference_hours":
        fifa_max_timezone_difference,

    "total_weather_exposure":
        fifa_total_weather_exposure,

    "average_weather_exposure":
        fifa_avg_weather_exposure,

    "maximum_weather_exposure":
        fifa_max_weather_exposure,

    "seating_opportunity":
        fifa_seating_opportunity
}

fifa_baseline_df = pd.DataFrame(
    fifa_baseline.items(),
    columns=["metric", "value"]
)

display(fifa_baseline_df)

# 4. Model 1 (Only Minimizing Travel Time)
## Model 1 minimizes total team travel time while keeping all group-stage matchups and match dates fixed. For each possible match–venue assignment, the travel-time coefficient includes round-trip travel for both participating teams because each team is assumed to return to its official base camp after every match. Candidate kickoff times are whole-hour local venue times from 10:00 to 22:00.

In [ ]:
# shared parameters (used in later models as well)

candidate_utc_datetime = {}

for m in Matches:
    for v in Venues:
        for t in Times:

            local_datetime = (
                pd.Timestamp(match_date[m])
                + pd.Timedelta(hours=t)
            )

            candidate_utc_datetime[(m, v, t)] = (
                local_datetime
                - pd.Timedelta(hours=venue_utc_offset[v])
            )

# numeric UTC kickoff time measured in hours from tournament start

tournament_start = min(Dates)

candidate_utc_hour = {}

for m in Matches:
    for v in Venues:
        for t in Times:

            candidate_utc_hour[(m, v, t)] = (
                candidate_utc_datetime[(m, v, t)]
                - tournament_start
            ).total_seconds() / 3600

In [ ]:
# model 1 specific parameters

model1_travel_time = {}

for m in Matches:
    for v in Venues:

        i = team_1[m]
        j = team_2[m]

        model1_travel_time[(m, v)] = (
            2 * travel_time[(i, v)]
            + 2 * travel_time[(j, v)]
        )

In [ ]:
# creating the model

model1 = gp.Model("Model_1_Travel_Time")

# decision variables

x = model1.addVars(
    Matches,
    Venues,
    Times,
    vtype=GRB.BINARY,
    name="x"
)

# selected UTC kickoff time for each match

kickoff_utc = model1.addVars(
    Matches,
    vtype=GRB.CONTINUOUS,
    name="kickoff_utc"
)

# objective function

model1.setObjective(
    gp.quicksum(
        model1_travel_time[(m, v)] * x[m, v, t]
        for m in Matches
        for v in Venues
        for t in Times
    ),
    GRB.MINIMIZE
)

# every match is assigned exactly once

for m in Matches:

    model1.addConstr(
        gp.quicksum(
            x[m, v, t]
            for v in Venues
            for t in Times
        ) == 1,
        name="MatchAssignment_" + str(m)
    )


# link each match to its selected UTC kickoff time

for m in Matches:

    model1.addConstr(
        kickoff_utc[m]
        ==
        gp.quicksum(
            candidate_utc_hour[(m, v, t)]
            * x[m, v, t]
            for v in Venues
            for t in Times
        ),
        name="KickoffTime_" + str(m)
    )


# at most one match per venue per local date

for v in Venues:
    for d in Dates:

        model1.addConstr(
            gp.quicksum(
                x[m, v, t]
                for m in matches_by_date[d]
                for t in Times
            ) <= 1,
            name="VenueDailyCapacity_"
            + str(v) + "_"
            + str(d)
        )

# minimum stadium usage

minimum_venue_matches = 3

for v in Venues:

    model1.addConstr(
        gp.quicksum(
            x[m, v, t]
            for m in Matches
            for t in Times
        ) >= minimum_venue_matches,
        name="MinimumVenueUsage_" + str(v)
    )

# round 1 and round 2 kickoff spacing

EarlyRoundMatches = (
    Round1Matches + Round2Matches
)

minimum_kickoff_gap = 3
big_M = 500

early_match_pairs = []

for index1 in range(len(EarlyRoundMatches)):
    for index2 in range(
        index1 + 1,
        len(EarlyRoundMatches)
    ):

        m1 = EarlyRoundMatches[index1]
        m2 = EarlyRoundMatches[index2]

        date_difference = abs(
            (
                match_date[m2]
                - match_date[m1]
            ).days
        )

        if date_difference <= 1:
            early_match_pairs.append(
                (m1, m2)
            )


early_order = model1.addVars(
    early_match_pairs,
    vtype=GRB.BINARY,
    name="early_order"
)


for m1, m2 in early_match_pairs:

    model1.addConstr(
        kickoff_utc[m2]
        - kickoff_utc[m1]
        >=
        minimum_kickoff_gap
        - big_M
        * (1 - early_order[m1, m2]),
        name="EarlySpacing1_"
        + str(m1) + "_"
        + str(m2)
    )

    model1.addConstr(
        kickoff_utc[m1]
        - kickoff_utc[m2]
        >=
        minimum_kickoff_gap
        - big_M
        * early_order[m1, m2],
        name="EarlySpacing2_"
        + str(m1) + "_"
        + str(m2)
    )


# check round 3 match pairs

for g in Groups:

    round3_matches = (
        matches_by_group_round[(g, 3)]
    )

    if len(round3_matches) != 2:
        raise ValueError(
            "Expected exactly 2 Round 3 matches for Group "
            + str(g)
        )


# round 3 simultaneous kickoff constraints

for g in Groups:

    round3_matches = (
        matches_by_group_round[(g, 3)]
    )

    m1 = round3_matches[0]
    m2 = round3_matches[1]

    model1.addConstr(
        kickoff_utc[m1]
        ==
        kickoff_utc[m2],
        name="Round3Simultaneous_"
        + str(g)
    )


# minimum 72-hour raw rest

minimum_raw_rest = 72

for i in Teams:

    matches_i = team_matches[i]

    for k in range(
        len(matches_i) - 1
    ):

        m1 = matches_i[k]
        m2 = matches_i[k + 1]

        model1.addConstr(
            kickoff_utc[m2]
            - kickoff_utc[m1]
            >= minimum_raw_rest,
            name="MinimumRest_"
            + str(i) + "_"
            + str(m1) + "_"
            + str(m2)
        )

# validate group-round structure

print()
print("Round 1 matches:", len(Round1Matches))
print("Round 2 matches:", len(Round2Matches))
print("Round 3 matches:", len(Round3Matches))
print()

# optimize model 1

model1.optimize()

# check model 1 solution status

if model1.Status == GRB.OPTIMAL:
    print("Model 1 optimal solution found.")
    print("Objective value:", model1.ObjVal)

elif model1.Status == GRB.INFEASIBLE:
    print("Model 1 is infeasible.")

else:
    print("Model 1 status:", model1.Status)

In [ ]:
# extracting model 1 optimized schedule

model1_schedule_rows = []

for m in Matches:

    for v in Venues:
        for t in Times:

            if x[m, v, t].X > 0.5:

                model1_schedule_rows.append({
                    "match_id": m,
                    "group": match_group[m],
                    "group_round": group_round[m],

                    "team_1_id": team_1[m],
                    "team_1_name":
                        team_name[team_1[m]],

                    "team_2_id": team_2[m],
                    "team_2_name":
                        team_name[team_2[m]],

                    "match_date":
                        match_date[m],

                    "venue_id": v,
                    "venue_name":
                        venue_name[v],

                    "venue_city":
                        venue_city[v],

                    "venue_country":
                        venue_country[v],

                    "kickoff_local_hour": t,

                    "kickoff_datetime_utc":
                        candidate_utc_datetime[
                            (m, v, t)
                        ]
                })


model1_schedule = pd.DataFrame(
    model1_schedule_rows
)

model1_schedule = (
    model1_schedule
    .sort_values(
        [
            "match_date",
            "kickoff_datetime_utc"
        ]
    )
    .reset_index(drop=True)
)

display(model1_schedule)

print(
    "Scheduled matches:",
    len(model1_schedule)
)

if len(model1_schedule) != 72:
    raise ValueError(
        "Expected 72 scheduled matches."
    )

# validate venue daily capacity

model1_venue_day_check = (
    model1_schedule
    .groupby(
        ["venue_id", "match_date"]
    )
    .size()
)

print(
    "Maximum matches at one venue "
    "on one date:",
    model1_venue_day_check.max()
)

# validate round 3 simultaneity

for g in Groups:

    round3_matches = (
        model1_schedule[
            (model1_schedule["group"] == g)
            &
            (
                model1_schedule[
                    "group_round"
                ] == 3
            )
        ]
    )

    kickoff_times = (
        round3_matches[
            "kickoff_datetime_utc"
        ].unique()
    )

    if len(kickoff_times) != 1:
        print(
            "Round 3 timing issue:",
            g
        )

In [ ]:
# model 1 performance metrics

# travel

model1_total_travel_time = 0

model1_team_travel = {
    i: 0
    for i in Teams
}

for _, row in model1_schedule.iterrows():

    i = row["team_1_id"]
    j = row["team_2_id"]
    v = row["venue_id"]

    team_1_travel = (
        2 * travel_time[(i, v)]
    )

    team_2_travel = (
        2 * travel_time[(j, v)]
    )

    model1_total_travel_time += (
        team_1_travel
        + team_2_travel
    )

    model1_team_travel[i] += (
        team_1_travel
    )

    model1_team_travel[j] += (
        team_2_travel
    )


model1_team_travel_df = pd.DataFrame({
    "team_id": Teams,

    "team_name": [
        team_name[i]
        for i in Teams
    ],

    "total_travel_time_hours": [
        model1_team_travel[i]
        for i in Teams
    ]
})


model1_avg_team_travel = (
    model1_team_travel_df[
        "total_travel_time_hours"
    ].mean()
)

model1_max_team_travel = (
    model1_team_travel_df[
        "total_travel_time_hours"
    ].max()
)

model1_min_team_travel = (
    model1_team_travel_df[
        "total_travel_time_hours"
    ].min()
)

model1_std_team_travel = (
    model1_team_travel_df[
        "total_travel_time_hours"
    ].std(ddof=0)
)

# rest and effective recovery

model1_recovery_rows = []

for i in Teams:

    team_schedule = model1_schedule[
        (model1_schedule["team_1_id"] == i)
        | (model1_schedule["team_2_id"] == i)
    ].copy()

    team_schedule = team_schedule.sort_values(
        "kickoff_datetime_utc"
    ).reset_index(drop=True)

    for k in range(len(team_schedule) - 1):

        previous_row = team_schedule.iloc[k]
        next_row = team_schedule.iloc[k + 1]

        raw_rest_hours = (
            next_row["kickoff_datetime_utc"]
            - previous_row["kickoff_datetime_utc"]
        ).total_seconds() / 3600

        previous_venue = (
            previous_row["venue_id"]
        )

        next_venue = (
            next_row["venue_id"]
        )

        recovery_travel_time = (
            travel_time[(i, previous_venue)]
            + travel_time[(i, next_venue)]
        )

        effective_recovery_hours = (
            raw_rest_hours
            - recovery_travel_time
        )

        model1_recovery_rows.append({
            "team_id": i,
            "team_name": team_name[i],
            "previous_match":
                previous_row["match_id"],
            "next_match":
                next_row["match_id"],
            "raw_rest_hours":
                raw_rest_hours,
            "recovery_travel_time_hours":
                recovery_travel_time,
            "effective_recovery_hours":
                effective_recovery_hours
        })


model1_recovery = pd.DataFrame(
    model1_recovery_rows
)

if len(model1_recovery) != 96:
    raise ValueError(
        "Expected 96 Model 1 recovery intervals."
    )

model1_min_raw_rest = (
    model1_recovery[
        "raw_rest_hours"
    ].min()
)

model1_avg_raw_rest = (
    model1_recovery[
        "raw_rest_hours"
    ].mean()
)

model1_min_effective_recovery = (
    model1_recovery[
        "effective_recovery_hours"
    ].min()
)

model1_avg_effective_recovery = (
    model1_recovery[
        "effective_recovery_hours"
    ].mean()
)

model1_intervals_below_72 = (
    model1_recovery[
        "raw_rest_hours"
    ] < 72
).sum()

# timezone difference

model1_timezone_rows = []

for _, row in model1_schedule.iterrows():

    m = row["match_id"]
    v = row["venue_id"]

    for i in [
        row["team_1_id"],
        row["team_2_id"]
    ]:

        model1_timezone_rows.append({
            "match_id": m,
            "team_id": i,
            "team_name": team_name[i],
            "venue_id": v,
            "timezone_difference_hours":
                timezone_difference[(i, v)]
        })


model1_timezone = pd.DataFrame(
    model1_timezone_rows
)

model1_avg_timezone_difference = (
    model1_timezone[
        "timezone_difference_hours"
    ].mean()
)

model1_max_timezone_difference = (
    model1_timezone[
        "timezone_difference_hours"
    ].max()
)

# weather exposure

missing_model1_weather = []

for _, row in model1_schedule.iterrows():

    v = row["venue_id"]
    d = row["match_date"]
    h = row["kickoff_local_hour"]

    if (v, d, h) not in weather_lookup:
        missing_model1_weather.append(
            (v, d, h)
        )

if len(missing_model1_weather) > 0:
    raise ValueError(
        "Missing weather data for Model 1 schedule."
    )


model1_weather_rows = []

for _, row in model1_schedule.iterrows():

    m = row["match_id"]
    v = row["venue_id"]
    d = row["match_date"]
    h = row["kickoff_local_hour"]

    heat_index = weather_lookup[
        (v, d, h)
    ]

    if weather_protected[v] == 1:
        weather_exposure = 0
    else:
        weather_exposure = heat_index

    model1_weather_rows.append({
        "match_id": m,
        "venue_id": v,
        "venue_name": venue_name[v],
        "date": d,
        "local_hour": h,
        "heat_index_f": heat_index,
        "weather_protected":
            weather_protected[v],
        "weather_exposure":
            weather_exposure
    })


model1_weather = pd.DataFrame(
    model1_weather_rows
)

model1_total_weather_exposure = (
    model1_weather[
        "weather_exposure"
    ].sum()
)

model1_avg_weather_exposure = (
    model1_weather[
        "weather_exposure"
    ].mean()
)

model1_max_weather_exposure = (
    model1_weather[
        "weather_exposure"
    ].max()
)

# seating opportunity

model1_seating_opportunity = 0

for _, row in model1_schedule.iterrows():

    v = row["venue_id"]

    model1_seating_opportunity += (
        venue_capacity[v]
    )

# venue and country usage

model1_venue_usage = {
    v: 0
    for v in Venues
}

for _, row in model1_schedule.iterrows():

    v = row["venue_id"]
    model1_venue_usage[v] += 1


model1_country_usage = {}

for _, row in model1_schedule.iterrows():

    v = row["venue_id"]
    country = venue_country[v]

    if country not in model1_country_usage:
        model1_country_usage[country] = 0

    model1_country_usage[country] += 1


model1_venue_usage_df = pd.DataFrame({
    "venue_id": Venues,

    "venue_name": [
        venue_name[v]
        for v in Venues
    ],

    "matches_hosted": [
        model1_venue_usage[v]
        for v in Venues
    ]
})


model1_country_usage_df = pd.DataFrame(
    model1_country_usage.items(),
    columns=[
        "country",
        "matches_hosted"
    ]
)

# model 1 summary dictionary

model1_summary = {
    "total_travel_time_hours":
        model1_total_travel_time,

    "average_team_travel_time_hours":
        model1_avg_team_travel,

    "maximum_team_travel_time_hours":
        model1_max_team_travel,

    "minimum_team_travel_time_hours":
        model1_min_team_travel,

    "travel_time_std_hours":
        model1_std_team_travel,

    "minimum_raw_rest_hours":
        model1_min_raw_rest,

    "average_raw_rest_hours":
        model1_avg_raw_rest,

    "minimum_effective_recovery_hours":
        model1_min_effective_recovery,

    "average_effective_recovery_hours":
        model1_avg_effective_recovery,

    "average_timezone_difference_hours":
        model1_avg_timezone_difference,

    "maximum_timezone_difference_hours":
        model1_max_timezone_difference,

    "total_weather_exposure":
        model1_total_weather_exposure,

    "average_weather_exposure":
        model1_avg_weather_exposure,

    "maximum_weather_exposure":
        model1_max_weather_exposure,

    "seating_opportunity":
        model1_seating_opportunity
}

# validation outputs

print(
    "Recovery intervals evaluated:",
    len(model1_recovery)
)

print(
    "Intervals below 72 hours raw rest:",
    model1_intervals_below_72
)

print(
    "Missing Model 1 weather lookups:",
    len(missing_model1_weather)
)

In [ ]:
# fifa baseline vs model 1

comparison_rows = []

for metric in fifa_baseline:

    fifa_value = fifa_baseline[metric]
    model_value = model1_summary[metric]

    difference = (
        model_value - fifa_value
    )

    if fifa_value != 0:
        percent_change = (
            difference / fifa_value
        ) * 100
    else:
        percent_change = np.nan

    comparison_rows.append({
        "metric": metric,
        "FIFA Baseline": fifa_value,
        "Model 1": model_value,
        "difference": difference,
        "percent_change": percent_change
    })


model1_comparison = pd.DataFrame(
    comparison_rows
)

display(
    model1_comparison.round(2)
)

# 5. Model 2 (Player Welfare Optimization)
## Model 2 focuses on player welfare through three objectives applied in priority order.

## 1. Maximize the minimum effective recovery time between consecutive matches.
## 2. Among schedules achieving the optimal minimum effective recovery, minimize total team time-zone displacement.
## 3. Among schedules preserving both of the previous optimal results, minimize total weather exposure.

## Effective recovery is defined as the raw time between consecutive match kickoffs minus the travel time from the previous match venue back to the team's base camp and from the base camp to the next match venue.

## The same fixed match dates, matchups, and tournament feasibility constraints used in Model 1 are retained.

In [ ]:
# model 2 specific parameters

# recovery intervals: each team has two recovery intervals between its three matches

RecoveryIntervals2 = []

for i in Teams:

    matches_i = team_matches[i]
    for k in range(len(matches_i) - 1):
        RecoveryIntervals2.append(
            (i, k)
        )


# weather exposure for each possible assignment

model2_weather_exposure = {}

for m in Matches:
    for v in Venues:
        for t in Times:

            if weather_protected[v] == 1:
                model2_weather_exposure[(m, v, t)] = 0

            else:
                model2_weather_exposure[(m, v, t)] = (
                    weather_lookup[
                        (v, match_date[m], t)
                    ]
                )


# creating model 2

model2 = gp.Model(
    "Model_2_Player_Welfare"
)


# decision variables

x2 = model2.addVars(
    Matches,
    Venues,
    Times,
    vtype=GRB.BINARY,
    name="x"
)


# selected UTC kickoff time for each match

kickoff_utc2 = model2.addVars(
    Matches,
    vtype=GRB.CONTINUOUS,
    name="kickoff_utc"
)


# effective recovery for each team interval

effective_recovery2 = model2.addVars(
    RecoveryIntervals2,
    lb=-GRB.INFINITY,
    vtype=GRB.CONTINUOUS,
    name="effective_recovery"
)


# minimum effective recovery across all intervals

minimum_effective_recovery2 = model2.addVar(
    lb=-GRB.INFINITY,
    vtype=GRB.CONTINUOUS,
    name="minimum_effective_recovery"
)


# total timezone displacement expression

model2_total_timezone = gp.quicksum(
    (
        timezone_difference[
            (team_1[m], v)
        ]
        +
        timezone_difference[
            (team_2[m], v)
        ]
    )
    * x2[m, v, t]

    for m in Matches
    for v in Venues
    for t in Times
)


# total weather exposure expression

model2_total_weather = gp.quicksum(
    model2_weather_exposure[(m, v, t)]
    * x2[m, v, t]

    for m in Matches
    for v in Venues
    for t in Times
)


# every match is assigned exactly once

for m in Matches:

    model2.addConstr(
        gp.quicksum(
            x2[m, v, t]
            for v in Venues
            for t in Times
        ) == 1,
        name="MatchAssignment_" + str(m)
    )


# link each match to its selected UTC kickoff time

for m in Matches:

    model2.addConstr(
        kickoff_utc2[m]
        ==
        gp.quicksum(
            candidate_utc_hour[(m, v, t)]
            * x2[m, v, t]

            for v in Venues
            for t in Times
        ),
        name="KickoffTime_" + str(m)
    )


# at most one match per venue per local date

for v in Venues:
    for d in Dates:

        model2.addConstr(
            gp.quicksum(
                x2[m, v, t]
                for m in matches_by_date[d]
                for t in Times
            ) <= 1,
            name="VenueDailyCapacity_"
            + str(v) + "_"
            + str(d)
        )


# minimum stadium usage

minimum_venue_matches = 3

for v in Venues:

    model2.addConstr(
        gp.quicksum(
            x2[m, v, t]
            for m in Matches
            for t in Times
        ) >= minimum_venue_matches,
        name="MinimumVenueUsage_"
        + str(v)
    )


# round 1 and round 2 kickoff spacing

EarlyRoundMatches2 = (
    Round1Matches + Round2Matches
)

minimum_kickoff_gap = 3
big_M = 500

early_match_pairs2 = []

for index1 in range(
    len(EarlyRoundMatches2)
):

    for index2 in range(
        index1 + 1,
        len(EarlyRoundMatches2)
    ):

        m1 = EarlyRoundMatches2[index1]
        m2 = EarlyRoundMatches2[index2]

        date_difference = abs(
            (
                match_date[m2]
                - match_date[m1]
            ).days
        )

        if date_difference <= 1:

            early_match_pairs2.append(
                (m1, m2)
            )


early_order2 = model2.addVars(
    early_match_pairs2,
    vtype=GRB.BINARY,
    name="early_order"
)


for m1, m2 in early_match_pairs2:

    model2.addConstr(
        kickoff_utc2[m2]
        - kickoff_utc2[m1]
        >=
        minimum_kickoff_gap
        - big_M
        * (
            1
            - early_order2[m1, m2]
        ),
        name="EarlySpacing1_"
        + str(m1) + "_"
        + str(m2)
    )

    model2.addConstr(
        kickoff_utc2[m1]
        - kickoff_utc2[m2]
        >=
        minimum_kickoff_gap
        - big_M
        * early_order2[m1, m2],
        name="EarlySpacing2_"
        + str(m1) + "_"
        + str(m2)
    )


# check round 3 match pairs

for g in Groups:

    round3_matches = (
        matches_by_group_round[(g, 3)]
    )

    if len(round3_matches) != 2:
        raise ValueError(
            "Expected exactly 2 Round 3 matches for Group "
            + str(g)
        )


# round 3 simultaneous kickoff constraints

for g in Groups:

    round3_matches = (
        matches_by_group_round[(g, 3)]
    )

    m1 = round3_matches[0]
    m2 = round3_matches[1]

    model2.addConstr(
        kickoff_utc2[m1]
        ==
        kickoff_utc2[m2],
        name="Round3Simultaneous_"
        + str(g)
    )


# minimum 72-hour raw rest

minimum_raw_rest = 72

for i in Teams:
    matches_i = team_matches[i]

    for k in range(len(matches_i) - 1):

        m1 = matches_i[k]
        m2 = matches_i[k + 1]

        model2.addConstr(
            kickoff_utc2[m2]
            - kickoff_utc2[m1]
            >= minimum_raw_rest,
            name="MinimumRest_"
            + str(i) + "_"
            + str(m1) + "_"
            + str(m2)
        )


# effective recovery constraints

for i, k in RecoveryIntervals2:

    matches_i = team_matches[i]

    m1 = matches_i[k]
    m2 = matches_i[k + 1]


    # travel from previous venue back to base camp

    travel_after_previous_match = (
        gp.quicksum(
            travel_time[(i, v)]
            * x2[m1, v, t]

            for v in Venues
            for t in Times
        )
    )


    # travel from base camp to next venue

    travel_before_next_match = (
        gp.quicksum(
            travel_time[(i, v)]
            * x2[m2, v, t]

            for v in Venues
            for t in Times
        )
    )


    model2.addConstr(
        effective_recovery2[i, k]
        ==
        kickoff_utc2[m2]
        - kickoff_utc2[m1]
        - travel_after_previous_match
        - travel_before_next_match,
        name="EffectiveRecovery_"
        + str(i) + "_"
        + str(k)
    )


# minimum effective recovery constraints

for i, k in RecoveryIntervals2:

    model2.addConstr(
        minimum_effective_recovery2
        <= effective_recovery2[i, k],
        name="MinimumEffectiveRecovery_"
        + str(i) + "_"
        + str(k)
    )


# validate structure

print(
    "Round 1 matches:",
    len(Round1Matches)
)

print(
    "Round 2 matches:",
    len(Round2Matches)
)

print(
    "Round 3 matches:",
    len(Round3Matches)
)

print(
    "Recovery intervals:",
    len(RecoveryIntervals2)
)

print()


# ------------------------------------------------------------
# stage 1
# maximize minimum effective recovery
# ------------------------------------------------------------

model2.setObjective(
    minimum_effective_recovery2,
    GRB.MAXIMIZE
)

model2.optimize()

if model2.Status != GRB.OPTIMAL:

    raise ValueError(
        "Model 2 Stage 1 did not solve optimally."
    )


best_minimum_recovery2 = (
    minimum_effective_recovery2.X
)

print()
print(
    "Stage 1 - Best minimum effective recovery:",
    round(
        best_minimum_recovery2,
        2
    ),
    "hours"
)

# lock stage 1 result

objective_tolerance = 0.0001

model2.addConstr(
    minimum_effective_recovery2
    >=
    best_minimum_recovery2
    - objective_tolerance,
    name="LockMinimumEffectiveRecovery"
)


# ------------------------------------------------------------
# stage 2
# minimize timezone displacement
# ------------------------------------------------------------

model2.setObjective(
    model2_total_timezone,
    GRB.MINIMIZE
)

model2.optimize()

if model2.Status != GRB.OPTIMAL:

    raise ValueError(
        "Model 2 Stage 2 did not solve optimally."
    )


best_timezone2 = (
    model2_total_timezone.getValue()
)

print(
    "Stage 2 - Best total timezone displacement:",
    round(
        best_timezone2,
        2
    ),
    "hours"
)


# lock stage 2 result

model2.addConstr(
    model2_total_timezone
    <=
    best_timezone2
    + objective_tolerance,
    name="LockTimezoneDisplacement"
)


# ------------------------------------------------------------
# stage 3
# minimize weather exposure
# ------------------------------------------------------------

model2.setObjective(
    model2_total_weather,
    GRB.MINIMIZE
)

model2.optimize()

if model2.Status != GRB.OPTIMAL:

    raise ValueError(
        "Model 2 Stage 3 did not solve optimally."
    )


best_weather2 = (
    model2_total_weather.getValue()
)


print(
    "Stage 3 - Best total weather exposure:",
    round(
        best_weather2,
        2
    )
)

print()

print(
    "Model 2 player welfare schedule found."
)

print(
    "Final minimum effective recovery:",
    round(
        minimum_effective_recovery2.X,
        2
    ),
    "hours"
)

print(
    "Final total timezone displacement:",
    round(
        model2_total_timezone.getValue(),
        2
    ),
    "hours"
)

print(
    "Final total weather exposure:",
    round(
        model2_total_weather.getValue(),
        2
    )
)

In [ ]:
# extracting model 2 optimized schedule

model2_schedule_rows = []

for m in Matches:

    for v in Venues:
        for t in Times:

            if x2[m, v, t].X > 0.5:

                model2_schedule_rows.append({
                    "match_id": m,

                    "group":
                        match_group[m],

                    "group_round":
                        group_round[m],

                    "team_1_id":
                        team_1[m],

                    "team_1_name":
                        team_name[
                            team_1[m]
                        ],

                    "team_2_id":
                        team_2[m],

                    "team_2_name":
                        team_name[
                            team_2[m]
                        ],

                    "match_date":
                        match_date[m],

                    "venue_id":
                        v,

                    "venue_name":
                        venue_name[v],

                    "venue_city":
                        venue_city[v],

                    "venue_country":
                        venue_country[v],

                    "kickoff_local_hour":
                        t,

                    "kickoff_datetime_utc":
                        candidate_utc_datetime[
                            (m, v, t)
                        ]
                })


model2_schedule = pd.DataFrame(
    model2_schedule_rows
)


model2_schedule = (
    model2_schedule
    .sort_values(
        [
            "match_date",
            "kickoff_datetime_utc"
        ]
    )
    .reset_index(drop=True)
)


display(model2_schedule)


# schedule validation

print(
    "Scheduled matches:",
    len(model2_schedule)
)

if len(model2_schedule) != 72:

    raise ValueError(
        "Expected 72 scheduled matches."
    )


# venue daily capacity

model2_venue_day_check = (
    model2_schedule
    .groupby(
        [
            "venue_id",
            "match_date"
        ]
    )
    .size()
)

print(
    "Maximum matches at one venue on one date:",
    model2_venue_day_check.max()
)


# minimum venue usage

model2_venue_counts = (
    model2_schedule[
        "venue_id"
    ]
    .value_counts()
    .reindex(
        Venues,
        fill_value=0
    )
)

print(
    "Minimum matches hosted by one venue:",
    model2_venue_counts.min()
)


# round 3 simultaneity

round3_issues2 = []

for g in Groups:

    round3_matches = model2_schedule[
        (
            model2_schedule[
                "group"
            ] == g
        )
        &
        (
            model2_schedule[
                "group_round"
            ] == 3
        )
    ]

    kickoff_times = (
        round3_matches[
            "kickoff_datetime_utc"
        ].unique()
    )

    if len(kickoff_times) != 1:

        round3_issues2.append(g)


print(
    "Round 3 simultaneity issues:",
    len(round3_issues2)
)

In [ ]:
# model 2 performance metrics

# travel

model2_total_travel_time = 0

model2_team_travel = {
    i: 0
    for i in Teams
}

for _, row in model2_schedule.iterrows():

    i = row["team_1_id"]
    j = row["team_2_id"]
    v = row["venue_id"]

    team_1_travel = (
        2 * travel_time[(i, v)]
    )

    team_2_travel = (
        2 * travel_time[(j, v)]
    )

    model2_total_travel_time += (
        team_1_travel
        + team_2_travel
    )

    model2_team_travel[i] += (
        team_1_travel
    )

    model2_team_travel[j] += (
        team_2_travel
    )


model2_team_travel_df = pd.DataFrame({
    "team_id":
        Teams,

    "team_name": [
        team_name[i]
        for i in Teams
    ],

    "total_travel_time_hours": [
        model2_team_travel[i]
        for i in Teams
    ]
})


model2_avg_team_travel = (
    model2_team_travel_df[
        "total_travel_time_hours"
    ].mean()
)

model2_max_team_travel = (
    model2_team_travel_df[
        "total_travel_time_hours"
    ].max()
)

model2_min_team_travel = (
    model2_team_travel_df[
        "total_travel_time_hours"
    ].min()
)

model2_std_team_travel = (
    model2_team_travel_df[
        "total_travel_time_hours"
    ].std(ddof=0)
)


# rest and effective recovery

model2_recovery_rows = []

for i in Teams:

    team_schedule = model2_schedule[
        (
            model2_schedule[
                "team_1_id"
            ] == i
        )
        |
        (
            model2_schedule[
                "team_2_id"
            ] == i
        )
    ].copy()


    team_schedule = (
        team_schedule
        .sort_values(
            "kickoff_datetime_utc"
        )
        .reset_index(drop=True)
    )


    for k in range(
        len(team_schedule) - 1
    ):

        previous_row = (
            team_schedule.iloc[k]
        )

        next_row = (
            team_schedule.iloc[k + 1]
        )


        raw_rest_hours = (
            next_row[
                "kickoff_datetime_utc"
            ]
            -
            previous_row[
                "kickoff_datetime_utc"
            ]
        ).total_seconds() / 3600


        previous_venue = (
            previous_row[
                "venue_id"
            ]
        )

        next_venue = (
            next_row[
                "venue_id"
            ]
        )


        recovery_travel_time = (
            travel_time[
                (i, previous_venue)
            ]
            +
            travel_time[
                (i, next_venue)
            ]
        )


        effective_recovery_hours = (
            raw_rest_hours
            - recovery_travel_time
        )


        model2_recovery_rows.append({
            "team_id":
                i,

            "team_name":
                team_name[i],

            "previous_match":
                previous_row[
                    "match_id"
                ],

            "next_match":
                next_row[
                    "match_id"
                ],

            "raw_rest_hours":
                raw_rest_hours,

            "recovery_travel_time_hours":
                recovery_travel_time,

            "effective_recovery_hours":
                effective_recovery_hours
        })


model2_recovery = pd.DataFrame(
    model2_recovery_rows
)


if len(model2_recovery) != 96:

    raise ValueError(
        "Expected 96 Model 2 recovery intervals."
    )


model2_min_raw_rest = (
    model2_recovery[
        "raw_rest_hours"
    ].min()
)

model2_avg_raw_rest = (
    model2_recovery[
        "raw_rest_hours"
    ].mean()
)

model2_min_effective_recovery = (
    model2_recovery[
        "effective_recovery_hours"
    ].min()
)

model2_avg_effective_recovery = (
    model2_recovery[
        "effective_recovery_hours"
    ].mean()
)

model2_intervals_below_72 = (
    model2_recovery[
        "raw_rest_hours"
    ] < 72
).sum()


# timezone difference

model2_timezone_rows = []

for _, row in model2_schedule.iterrows():

    m = row["match_id"]
    v = row["venue_id"]

    for i in [
        row["team_1_id"],
        row["team_2_id"]
    ]:

        model2_timezone_rows.append({
            "match_id":
                m,

            "team_id":
                i,

            "team_name":
                team_name[i],

            "venue_id":
                v,

            "timezone_difference_hours":
                timezone_difference[
                    (i, v)
                ]
        })


model2_timezone = pd.DataFrame(
    model2_timezone_rows
)


model2_avg_timezone_difference = (
    model2_timezone[
        "timezone_difference_hours"
    ].mean()
)

model2_max_timezone_difference = (
    model2_timezone[
        "timezone_difference_hours"
    ].max()
)


# weather exposure

missing_model2_weather = []

for _, row in model2_schedule.iterrows():

    v = row["venue_id"]
    d = row["match_date"]
    h = row["kickoff_local_hour"]

    if (
        v,
        d,
        h
    ) not in weather_lookup:

        missing_model2_weather.append(
            (v, d, h)
        )


if len(missing_model2_weather) > 0:

    raise ValueError(
        "Missing weather data for Model 2 schedule."
    )


model2_weather_rows = []

for _, row in model2_schedule.iterrows():

    m = row["match_id"]
    v = row["venue_id"]
    d = row["match_date"]
    h = row["kickoff_local_hour"]

    heat_index = weather_lookup[
        (v, d, h)
    ]

    if weather_protected[v] == 1:

        weather_exposure = 0

    else:

        weather_exposure = (
            heat_index
        )


    model2_weather_rows.append({
        "match_id":
            m,

        "venue_id":
            v,

        "venue_name":
            venue_name[v],

        "date":
            d,

        "local_hour":
            h,

        "heat_index_f":
            heat_index,

        "weather_protected":
            weather_protected[v],

        "weather_exposure":
            weather_exposure
    })


model2_weather = pd.DataFrame(
    model2_weather_rows
)


model2_total_weather_exposure = (
    model2_weather[
        "weather_exposure"
    ].sum()
)

model2_avg_weather_exposure = (
    model2_weather[
        "weather_exposure"
    ].mean()
)

model2_max_weather_exposure = (
    model2_weather[
        "weather_exposure"
    ].max()
)


# seating opportunity

model2_seating_opportunity = 0

for _, row in model2_schedule.iterrows():

    v = row["venue_id"]

    model2_seating_opportunity += (
        venue_capacity[v]
    )


# venue and country usage

model2_venue_usage = {
    v: 0
    for v in Venues
}

for _, row in model2_schedule.iterrows():

    v = row["venue_id"]

    model2_venue_usage[v] += 1


model2_country_usage = {}

for _, row in model2_schedule.iterrows():

    v = row["venue_id"]

    country = (
        venue_country[v]
    )

    if country not in model2_country_usage:

        model2_country_usage[
            country
        ] = 0

    model2_country_usage[
        country
    ] += 1


model2_venue_usage_df = pd.DataFrame({
    "venue_id":
        Venues,

    "venue_name": [
        venue_name[v]
        for v in Venues
    ],

    "matches_hosted": [
        model2_venue_usage[v]
        for v in Venues
    ]
})


model2_country_usage_df = pd.DataFrame(
    model2_country_usage.items(),
    columns=[
        "country",
        "matches_hosted"
    ]
)


# model 2 summary

model2_summary = {
    "total_travel_time_hours":
        model2_total_travel_time,

    "average_team_travel_time_hours":
        model2_avg_team_travel,

    "maximum_team_travel_time_hours":
        model2_max_team_travel,

    "minimum_team_travel_time_hours":
        model2_min_team_travel,

    "travel_time_std_hours":
        model2_std_team_travel,

    "minimum_raw_rest_hours":
        model2_min_raw_rest,

    "average_raw_rest_hours":
        model2_avg_raw_rest,

    "minimum_effective_recovery_hours":
        model2_min_effective_recovery,

    "average_effective_recovery_hours":
        model2_avg_effective_recovery,

    "average_timezone_difference_hours":
        model2_avg_timezone_difference,

    "maximum_timezone_difference_hours":
        model2_max_timezone_difference,

    "total_weather_exposure":
        model2_total_weather_exposure,

    "average_weather_exposure":
        model2_avg_weather_exposure,

    "maximum_weather_exposure":
        model2_max_weather_exposure,

    "seating_opportunity":
        model2_seating_opportunity
}


# validation outputs

print(
    "Recovery intervals evaluated:",
    len(model2_recovery)
)

print(
    "Intervals below 72 hours raw rest:",
    model2_intervals_below_72
)

print(
    "Missing Model 2 weather lookups:",
    len(missing_model2_weather)
)

In [ ]:
# fifa baseline vs model 1 vs model 2

comparison_rows = []

for metric in fifa_baseline:

    fifa_value = (
        fifa_baseline[metric]
    )

    model1_value = (
        model1_summary[metric]
    )

    model2_value = (
        model2_summary[metric]
    )


    if fifa_value != 0:

        model1_vs_fifa_pct = (
            (
                model1_value
                - fifa_value
            )
            / fifa_value
        ) * 100

        model2_vs_fifa_pct = (
            (
                model2_value
                - fifa_value
            )
            / fifa_value
        ) * 100

    else:

        model1_vs_fifa_pct = np.nan
        model2_vs_fifa_pct = np.nan


    comparison_rows.append({
        "metric":
            metric,

        "FIFA Baseline":
            fifa_value,

        "Model 1":
            model1_value,

        "Model 1 vs FIFA (%)":
            model1_vs_fifa_pct,

        "Model 2":
            model2_value,

        "Model 2 vs FIFA (%)":
            model2_vs_fifa_pct
    })


model2_comparison = pd.DataFrame(
    comparison_rows
)


display(
    model2_comparison.round(2)
)

# 5. Model 3 (Commercial and Broadcast Optimization)
## Model 3 focuses on FIFA's commercial interests through two objectives applied in priority order.
## 1. Minimize total broadcast inconvenience across global audience regions and the home markets of the two teams playing each match.
## 2. Among schedules achieving the optimal broadcast result, maximize total seating opportunity.
## A local viewing time between 10:00 and 22:00 receives no broadcast penalty. For times outside this range, one penalty point is added for every hour outside the acceptable viewing window.
## The global broadcast component reflects the distribution and importance of audience regions, while the home-market component reflects the local viewing times of the two teams actually playing each match. Equal weight is initially given to the two components. 
## The optimized schedule must retain at least 95% of FIFA's baseline seating opportunity. The same fixed match dates, matchups, and tournament feasibility constraints used in Models 1 and 2 are retained.

In [ ]:
# model 3 specific parameters


# home-market UTC offsets

home_utc_offset = dict(zip(
    teams["team_id"],
    teams["home_utc_offset"]
))


# viewing penalty function

def viewing_penalty(h):

    h = h % 24

    if h < 10:
        return 10 - h

    elif h > 22:
        return h - 22

    else:
        return 0


# global audience penalty for each UTC hour

global_broadcast_penalty = {}

for utc_hour in range(24):

    penalty = 0

    for _, row in audience_regions.iterrows():

        timezone_share = (
            row["timezone_population"]
            /
            row["regional_population"]
        )

        region_local_hour = (
            utc_hour
            + row["utc_offset"]
        ) % 24

        penalty += (
            row["regional_viewership"]
            * timezone_share
            * viewing_penalty(
                region_local_hour
            )
        )

    global_broadcast_penalty[
        utc_hour
    ] = penalty


# home-market penalty for each match and UTC hour

home_market_penalty = {}

for m in Matches:

    i = team_1[m]
    j = team_2[m]

    for utc_hour in range(24):

        team_1_home_hour = (
            utc_hour
            + home_utc_offset[i]
        ) % 24

        team_2_home_hour = (
            utc_hour
            + home_utc_offset[j]
        ) % 24

        home_market_penalty[
            (m, utc_hour)
        ] = (
            viewing_penalty(
                team_1_home_hour
            )
            +
            viewing_penalty(
                team_2_home_hour
            )
        ) / 2


# combined broadcast penalty for every possible assignment

alpha = 0.5

model3_broadcast_penalty = {}

for m in Matches:

    for v in Venues:

        for t in Times:

            utc_hour = (
                candidate_utc_datetime[
                    (m, v, t)
                ].hour
            )

            model3_broadcast_penalty[
                (m, v, t)
            ] = (
                alpha
                * global_broadcast_penalty[
                    utc_hour
                ]
                +
                (1 - alpha)
                * home_market_penalty[
                    (m, utc_hour)
                ]
            )


# creating model 3

model3 = gp.Model(
    "Model_3_Commercial_Broadcast"
)


# decision variables

x3 = model3.addVars(
    Matches,
    Venues,
    Times,
    vtype=GRB.BINARY,
    name="x"
)


# selected UTC kickoff time for each match

kickoff_utc3 = model3.addVars(
    Matches,
    vtype=GRB.CONTINUOUS,
    name="kickoff_utc"
)


# objective expressions


# total broadcast penalty

model3_total_broadcast_penalty = (
    gp.quicksum(
        model3_broadcast_penalty[
            (m, v, t)
        ]
        * x3[m, v, t]

        for m in Matches
        for v in Venues
        for t in Times
    )
)


# total seating opportunity

model3_seating_opportunity = (
    gp.quicksum(
        venue_capacity[v]
        * x3[m, v, t]

        for m in Matches
        for v in Venues
        for t in Times
    )
)


# every match is assigned exactly once

for m in Matches:

    model3.addConstr(
        gp.quicksum(
            x3[m, v, t]

            for v in Venues
            for t in Times
        ) == 1,

        name="MatchAssignment_"
        + str(m)
    )


# link each match to its selected UTC kickoff time

for m in Matches:

    model3.addConstr(
        kickoff_utc3[m]
        ==
        gp.quicksum(
            candidate_utc_hour[
                (m, v, t)
            ]
            * x3[m, v, t]

            for v in Venues
            for t in Times
        ),

        name="KickoffTime_"
        + str(m)
    )


# at most one match per venue per local date

for v in Venues:

    for d in Dates:

        model3.addConstr(
            gp.quicksum(
                x3[m, v, t]

                for m in matches_by_date[d]
                for t in Times
            ) <= 1,

            name="VenueDailyCapacity_"
            + str(v) + "_"
            + str(d)
        )


# minimum stadium usage

minimum_venue_matches = 3

for v in Venues:

    model3.addConstr(
        gp.quicksum(
            x3[m, v, t]

            for m in Matches
            for t in Times
        )
        >= minimum_venue_matches,

        name="MinimumVenueUsage_"
        + str(v)
    )


# minimum seating retention

minimum_seating_retention = 0.95

model3.addConstr(
    model3_seating_opportunity
    >=
    minimum_seating_retention
    * fifa_baseline[
        "seating_opportunity"
    ],

    name="MinimumSeatingRetention"
)


# round 1 and round 2 kickoff spacing

EarlyRoundMatches3 = (
    Round1Matches
    + Round2Matches
)

minimum_kickoff_gap = 3

big_M = 500

early_match_pairs3 = []

for index1 in range(
    len(EarlyRoundMatches3)
):

    for index2 in range(
        index1 + 1,
        len(EarlyRoundMatches3)
    ):

        m1 = EarlyRoundMatches3[index1]

        m2 = EarlyRoundMatches3[index2]

        date_difference = abs(
            (
                match_date[m2]
                - match_date[m1]
            ).days
        )

        if date_difference <= 1:

            early_match_pairs3.append(
                (m1, m2)
            )


early_order3 = model3.addVars(
    early_match_pairs3,
    vtype=GRB.BINARY,
    name="early_order"
)


for m1, m2 in early_match_pairs3:

    model3.addConstr(
        kickoff_utc3[m2]
        - kickoff_utc3[m1]
        >=
        minimum_kickoff_gap
        -
        big_M
        * (
            1
            - early_order3[
                m1, m2
            ]
        ),

        name="EarlySpacing1_"
        + str(m1) + "_"
        + str(m2)
    )


    model3.addConstr(
        kickoff_utc3[m1]
        - kickoff_utc3[m2]
        >=
        minimum_kickoff_gap
        -
        big_M
        * early_order3[
            m1, m2
        ],

        name="EarlySpacing2_"
        + str(m1) + "_"
        + str(m2)
    )


# check round 3 match pairs

for g in Groups:

    round3_matches = (
        matches_by_group_round[
            (g, 3)
        ]
    )

    if len(round3_matches) != 2:

        raise ValueError(
            "Expected exactly 2 Round 3 matches for Group "
            + str(g)
        )


# round 3 simultaneous kickoff constraints

for g in Groups:

    round3_matches = (
        matches_by_group_round[
            (g, 3)
        ]
    )

    m1 = round3_matches[0]
    m2 = round3_matches[1]

    model3.addConstr(
        kickoff_utc3[m1]
        ==
        kickoff_utc3[m2],

        name="Round3Simultaneous_"
        + str(g)
    )


# minimum 72-hour raw rest

minimum_raw_rest = 72

for i in Teams:

    matches_i = team_matches[i]

    for k in range(
        len(matches_i) - 1
    ):

        m1 = matches_i[k]
        m2 = matches_i[k + 1]

        model3.addConstr(
            kickoff_utc3[m2]
            - kickoff_utc3[m1]
            >= minimum_raw_rest,

            name="MinimumRest_"
            + str(i) + "_"
            + str(m1) + "_"
            + str(m2)
        )


# validate structure

print(
    "Round 1 matches:",
    len(Round1Matches)
)

print(
    "Round 2 matches:",
    len(Round2Matches)
)

print(
    "Round 3 matches:",
    len(Round3Matches)
)

print(
    "FIFA seating opportunity:",
    fifa_baseline[
        "seating_opportunity"
    ]
)

print(
    "Minimum allowed Model 3 seating:",
    round(
        0.95
        * fifa_baseline[
            "seating_opportunity"
        ],
        2
    )
)

print()


# ------------------------------------------------------------
# stage 1
# minimize broadcast penalty
# ------------------------------------------------------------

model3.setObjective(
    model3_total_broadcast_penalty,
    GRB.MINIMIZE
)

model3.optimize()


if model3.Status != GRB.OPTIMAL:

    raise ValueError(
        "Model 3 Stage 1 did not solve optimally."
    )


best_broadcast_penalty3 = (
    model3_total_broadcast_penalty
    .getValue()
)


print()

print(
    "Stage 1 - Best total broadcast penalty:",
    round(
        best_broadcast_penalty3,
        4
    )
)


# lock stage 1 result

broadcast_tolerance = (
    0.005
    * best_broadcast_penalty3
)

model3.addConstr(
    model3_total_broadcast_penalty
    <=
    best_broadcast_penalty3
    + broadcast_tolerance,
    name="LockBroadcastPenalty"
)


# ------------------------------------------------------------
# stage 2
# maximize seating opportunity
# ------------------------------------------------------------

model3.setObjective(
    model3_seating_opportunity,
    GRB.MAXIMIZE
)

model3.optimize()


if model3.Status != GRB.OPTIMAL:

    raise ValueError(
        "Model 3 Stage 2 did not solve optimally."
    )


best_seating_opportunity3 = (
    model3_seating_opportunity
    .getValue()
)


print(
    "Stage 2 - Best seating opportunity:",
    round(
        best_seating_opportunity3,
        2
    )
)

print()

print(
    "Model 3 commercial schedule found."
)

print(
    "Final total broadcast penalty:",
    round(
        model3_total_broadcast_penalty
        .getValue(),
        4
    )
)

print(
    "Final seating opportunity:",
    round(
        model3_seating_opportunity
        .getValue(),
        2
    )
)

In [ ]:
# extracting model 3 optimized schedule

model3_schedule_rows = []

for m in Matches:

    for v in Venues:

        for t in Times:

            if x3[m, v, t].X > 0.5:

                model3_schedule_rows.append({

                    "match_id":
                        m,

                    "group":
                        match_group[m],

                    "group_round":
                        group_round[m],

                    "team_1_id":
                        team_1[m],

                    "team_1_name":
                        team_name[
                            team_1[m]
                        ],

                    "team_2_id":
                        team_2[m],

                    "team_2_name":
                        team_name[
                            team_2[m]
                        ],

                    "match_date":
                        match_date[m],

                    "venue_id":
                        v,

                    "venue_name":
                        venue_name[v],

                    "venue_city":
                        venue_city[v],

                    "venue_country":
                        venue_country[v],

                    "kickoff_local_hour":
                        t,

                    "kickoff_datetime_utc":
                        candidate_utc_datetime[
                            (m, v, t)
                        ],

                    "broadcast_penalty":
                        model3_broadcast_penalty[
                            (m, v, t)
                        ]
                })


model3_schedule = pd.DataFrame(
    model3_schedule_rows
)


model3_schedule = (
    model3_schedule
    .sort_values(
        [
            "match_date",
            "kickoff_datetime_utc"
        ]
    )
    .reset_index(drop=True)
)


display(
    model3_schedule
)


# schedule validation

print(
    "Scheduled matches:",
    len(model3_schedule)
)


if len(model3_schedule) != 72:

    raise ValueError(
        "Expected 72 scheduled matches."
    )


# venue daily capacity

model3_venue_day_check = (
    model3_schedule
    .groupby(
        [
            "venue_id",
            "match_date"
        ]
    )
    .size()
)


print(
    "Maximum matches at one venue on one date:",
    model3_venue_day_check.max()
)


# minimum venue usage

model3_venue_counts = (
    model3_schedule[
        "venue_id"
    ]
    .value_counts()
    .reindex(
        Venues,
        fill_value=0
    )
)


print(
    "Minimum matches hosted by one venue:",
    model3_venue_counts.min()
)


# seating retention

model3_extracted_seating = 0

for _, row in model3_schedule.iterrows():

    model3_extracted_seating += (
        venue_capacity[
            row["venue_id"]
        ]
    )


print(
    "Seating retention vs FIFA (%):",
    round(
        (
            model3_extracted_seating
            /
            fifa_baseline[
                "seating_opportunity"
            ]
        )
        * 100,
        2
    )
)


# round 3 simultaneity

round3_issues3 = []

for g in Groups:

    round3_matches = (
        model3_schedule[
            (
                model3_schedule[
                    "group"
                ] == g
            )
            &
            (
                model3_schedule[
                    "group_round"
                ] == 3
            )
        ]
    )


    kickoff_times = (
        round3_matches[
            "kickoff_datetime_utc"
        ]
        .unique()
    )


    if len(kickoff_times) != 1:

        round3_issues3.append(g)


print(
    "Round 3 simultaneity issues:",
    len(round3_issues3)
)

In [ ]:
# model 3 performance metrics


# travel

model3_total_travel_time = 0

model3_team_travel = {
    i: 0
    for i in Teams
}


for _, row in model3_schedule.iterrows():

    i = row["team_1_id"]
    j = row["team_2_id"]
    v = row["venue_id"]

    team_1_travel = (
        2 * travel_time[(i, v)]
    )

    team_2_travel = (
        2 * travel_time[(j, v)]
    )

    model3_total_travel_time += (
        team_1_travel
        + team_2_travel
    )

    model3_team_travel[i] += (
        team_1_travel
    )

    model3_team_travel[j] += (
        team_2_travel
    )


model3_team_travel_df = pd.DataFrame({

    "team_id":
        Teams,

    "team_name": [
        team_name[i]
        for i in Teams
    ],

    "total_travel_time_hours": [
        model3_team_travel[i]
        for i in Teams
    ]
})


model3_avg_team_travel = (
    model3_team_travel_df[
        "total_travel_time_hours"
    ].mean()
)

model3_max_team_travel = (
    model3_team_travel_df[
        "total_travel_time_hours"
    ].max()
)

model3_min_team_travel = (
    model3_team_travel_df[
        "total_travel_time_hours"
    ].min()
)

model3_std_team_travel = (
    model3_team_travel_df[
        "total_travel_time_hours"
    ].std(
        ddof=0
    )
)


# rest and effective recovery

model3_recovery_rows = []


for i in Teams:

    team_schedule = (
        model3_schedule[
            (
                model3_schedule[
                    "team_1_id"
                ] == i
            )
            |
            (
                model3_schedule[
                    "team_2_id"
                ] == i
            )
        ]
        .copy()
    )


    team_schedule = (
        team_schedule
        .sort_values(
            "kickoff_datetime_utc"
        )
        .reset_index(
            drop=True
        )
    )


    for k in range(
        len(team_schedule) - 1
    ):

        previous_row = (
            team_schedule.iloc[k]
        )

        next_row = (
            team_schedule.iloc[k + 1]
        )


        raw_rest_hours = (
            next_row[
                "kickoff_datetime_utc"
            ]
            -
            previous_row[
                "kickoff_datetime_utc"
            ]
        ).total_seconds() / 3600


        previous_venue = (
            previous_row[
                "venue_id"
            ]
        )

        next_venue = (
            next_row[
                "venue_id"
            ]
        )


        recovery_travel_time = (
            travel_time[
                (
                    i,
                    previous_venue
                )
            ]
            +
            travel_time[
                (
                    i,
                    next_venue
                )
            ]
        )


        effective_recovery_hours = (
            raw_rest_hours
            -
            recovery_travel_time
        )


        model3_recovery_rows.append({

            "team_id":
                i,

            "team_name":
                team_name[i],

            "previous_match":
                previous_row[
                    "match_id"
                ],

            "next_match":
                next_row[
                    "match_id"
                ],

            "raw_rest_hours":
                raw_rest_hours,

            "recovery_travel_time_hours":
                recovery_travel_time,

            "effective_recovery_hours":
                effective_recovery_hours
        })


model3_recovery = pd.DataFrame(
    model3_recovery_rows
)


if len(model3_recovery) != 96:

    raise ValueError(
        "Expected 96 Model 3 recovery intervals."
    )


model3_min_raw_rest = (
    model3_recovery[
        "raw_rest_hours"
    ].min()
)

model3_avg_raw_rest = (
    model3_recovery[
        "raw_rest_hours"
    ].mean()
)

model3_min_effective_recovery = (
    model3_recovery[
        "effective_recovery_hours"
    ].min()
)

model3_avg_effective_recovery = (
    model3_recovery[
        "effective_recovery_hours"
    ].mean()
)

model3_intervals_below_72 = (
    model3_recovery[
        "raw_rest_hours"
    ]
    < 72
).sum()


# timezone difference

model3_timezone_rows = []


for _, row in model3_schedule.iterrows():

    m = row["match_id"]
    v = row["venue_id"]

    for i in [
        row["team_1_id"],
        row["team_2_id"]
    ]:

        model3_timezone_rows.append({

            "match_id":
                m,

            "team_id":
                i,

            "team_name":
                team_name[i],

            "venue_id":
                v,

            "timezone_difference_hours":
                timezone_difference[
                    (i, v)
                ]
        })


model3_timezone = pd.DataFrame(
    model3_timezone_rows
)


model3_avg_timezone_difference = (
    model3_timezone[
        "timezone_difference_hours"
    ].mean()
)

model3_max_timezone_difference = (
    model3_timezone[
        "timezone_difference_hours"
    ].max()
)


# weather exposure

missing_model3_weather = []


for _, row in model3_schedule.iterrows():

    v = row["venue_id"]
    d = row["match_date"]
    h = row["kickoff_local_hour"]

    if (
        v,
        d,
        h
    ) not in weather_lookup:

        missing_model3_weather.append(
            (
                v,
                d,
                h
            )
        )


if len(
    missing_model3_weather
) > 0:

    raise ValueError(
        "Missing weather data for Model 3 schedule."
    )


model3_weather_rows = []


for _, row in model3_schedule.iterrows():

    m = row["match_id"]
    v = row["venue_id"]
    d = row["match_date"]
    h = row["kickoff_local_hour"]

    heat_index = (
        weather_lookup[
            (v, d, h)
        ]
    )


    if weather_protected[v] == 1:

        weather_exposure = 0

    else:

        weather_exposure = (
            heat_index
        )


    model3_weather_rows.append({

        "match_id":
            m,

        "venue_id":
            v,

        "venue_name":
            venue_name[v],

        "date":
            d,

        "local_hour":
            h,

        "heat_index_f":
            heat_index,

        "weather_protected":
            weather_protected[v],

        "weather_exposure":
            weather_exposure
    })


model3_weather = pd.DataFrame(
    model3_weather_rows
)


model3_total_weather_exposure = (
    model3_weather[
        "weather_exposure"
    ].sum()
)

model3_avg_weather_exposure = (
    model3_weather[
        "weather_exposure"
    ].mean()
)

model3_max_weather_exposure = (
    model3_weather[
        "weather_exposure"
    ].max()
)


# seating opportunity

model3_total_seating_opportunity = 0


for _, row in model3_schedule.iterrows():

    v = row["venue_id"]

    model3_total_seating_opportunity += (
        venue_capacity[v]
    )


# broadcast penalty

model3_total_broadcast_penalty_metric = (
    model3_schedule[
        "broadcast_penalty"
    ].sum()
)

model3_average_broadcast_penalty = (
    model3_schedule[
        "broadcast_penalty"
    ].mean()
)

model3_maximum_broadcast_penalty = (
    model3_schedule[
        "broadcast_penalty"
    ].max()
)


# venue and country usage

model3_venue_usage = {
    v: 0
    for v in Venues
}


for _, row in model3_schedule.iterrows():

    v = row["venue_id"]

    model3_venue_usage[v] += 1


model3_country_usage = {}


for _, row in model3_schedule.iterrows():

    v = row["venue_id"]

    country = (
        venue_country[v]
    )

    if country not in model3_country_usage:

        model3_country_usage[
            country
        ] = 0

    model3_country_usage[
        country
    ] += 1


model3_venue_usage_df = pd.DataFrame({

    "venue_id":
        Venues,

    "venue_name": [
        venue_name[v]
        for v in Venues
    ],

    "matches_hosted": [
        model3_venue_usage[v]
        for v in Venues
    ]
})


model3_country_usage_df = pd.DataFrame(
    model3_country_usage.items(),

    columns=[
        "country",
        "matches_hosted"
    ]
)


# model 3 summary

model3_summary = {

    "total_travel_time_hours":
        model3_total_travel_time,

    "average_team_travel_time_hours":
        model3_avg_team_travel,

    "maximum_team_travel_time_hours":
        model3_max_team_travel,

    "minimum_team_travel_time_hours":
        model3_min_team_travel,

    "travel_time_std_hours":
        model3_std_team_travel,

    "minimum_raw_rest_hours":
        model3_min_raw_rest,

    "average_raw_rest_hours":
        model3_avg_raw_rest,

    "minimum_effective_recovery_hours":
        model3_min_effective_recovery,

    "average_effective_recovery_hours":
        model3_avg_effective_recovery,

    "average_timezone_difference_hours":
        model3_avg_timezone_difference,

    "maximum_timezone_difference_hours":
        model3_max_timezone_difference,

    "total_weather_exposure":
        model3_total_weather_exposure,

    "average_weather_exposure":
        model3_avg_weather_exposure,

    "maximum_weather_exposure":
        model3_max_weather_exposure,

    "seating_opportunity":
        model3_total_seating_opportunity
}


# validation outputs

print(
    "Recovery intervals evaluated:",
    len(model3_recovery)
)

print(
    "Intervals below 72 hours raw rest:",
    model3_intervals_below_72
)

print(
    "Missing Model 3 weather lookups:",
    len(missing_model3_weather)
)

print(
    "Total broadcast penalty:",
    round(
        model3_total_broadcast_penalty_metric,
        4
    )
)

print(
    "Average broadcast penalty:",
    round(
        model3_average_broadcast_penalty,
        4
    )
)

print(
    "Maximum match broadcast penalty:",
    round(
        model3_maximum_broadcast_penalty,
        4
    )
)

In [ ]:
# fifa baseline vs model 1 vs model 2 vs model 3

comparison_rows = []


for metric in fifa_baseline:

    fifa_value = (
        fifa_baseline[
            metric
        ]
    )

    model1_value = (
        model1_summary[
            metric
        ]
    )

    model2_value = (
        model2_summary[
            metric
        ]
    )

    model3_value = (
        model3_summary[
            metric
        ]
    )


    if fifa_value != 0:

        model1_vs_fifa_pct = (
            (
                model1_value
                - fifa_value
            )
            / fifa_value
        ) * 100

        model2_vs_fifa_pct = (
            (
                model2_value
                - fifa_value
            )
            / fifa_value
        ) * 100

        model3_vs_fifa_pct = (
            (
                model3_value
                - fifa_value
            )
            / fifa_value
        ) * 100

    else:

        model1_vs_fifa_pct = np.nan

        model2_vs_fifa_pct = np.nan

        model3_vs_fifa_pct = np.nan


    comparison_rows.append({

        "metric":
            metric,

        "FIFA Baseline":
            fifa_value,

        "Model 1":
            model1_value,

        "Model 1 vs FIFA (%)":
            model1_vs_fifa_pct,

        "Model 2":
            model2_value,

        "Model 2 vs FIFA (%)":
            model2_vs_fifa_pct,

        "Model 3":
            model3_value,

        "Model 3 vs FIFA (%)":
            model3_vs_fifa_pct
    })


model3_comparison = pd.DataFrame(
    comparison_rows
)


display(
    model3_comparison.round(2)
)